# Ylivertainen v2 — Clinical Association Pipeline

End-to-end workflow for finding statistically valid associations between **target outcomes**
and **predictor variables** in a clinical dataset.

This notebook drives six universal modules:

| Module                       | Purpose                                                    |
|------------------------------|------------------------------------------------------------|
| `schema_infer.py`            | Auto-classify each column (continuous, ordinal, …)         |
| `cleaning.py`                | Apply the schema, audit duplicates, derive new columns     |
| `dda.py`                     | Per-column descriptive stats + SVG plots                   |
| `missingness_resolution.py`  | Missing pattern analysis, flags, MICE multiple imputation  |
| `eda.py`                     | Univariate target × predictor screening (FDR-corrected)    |
| `inferential.py`             | Multivariable logistic regression with Rubin pooling       |

**Pipeline order**

```
load → infer schema → clean → DDA → missingness → derive new cols → DDA again
   → EDA screen → MICE impute → multivariable logistic (Rubin pool) → outputs
```

All outputs land under `output/<stage>/{figures,tables}/` as SVG and CSV.


## 0. Setup

In [1]:
import warnings
from sklearn.exceptions import ConvergenceWarning
import pandas as pd
pd.set_option("display.max_columns", None)

import numpy as np
import shutil
from pathlib import Path

from schema_infer import (infer_schema, print_schema_template, print_column_uniques, schema_summary,
                          export_schema_summary, ColSpec)
from cleaning import (apply_schema, audit_duplicates, export_cleaning_artifacts,
                      write_cleaned_csv, bin_numeric, bin_datetime, make_missing_flag,
                      combine_categories)
from dda import run_dda
from missingness_resolution import (analyze_missingness, add_missing_flags,
                                    mark_structural_missing, drop_rows,
                                    mice_impute, simple_impute, imputation_audit)
from eda import screen_associations
from diagnostic_accuracy import screen_diagnostic_accuracy
from inferential import run_inferential, summarize_multivariable_cases

OUTPUT_ROOT = Path("output")
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

from config import load

#🟧🟧🟧 None = all years; e.g. [2025] for one cohort year

ANALYSIS_YEARS: list[int] | None = None

# Run top to bottom

Each section: **edit → run → next**. No jumping back to a config block at the top.


## 01. Load data


In [2]:
DATA_PATH = "Meningiomas PSKUS grants - Visi pacienti.csv"   # or "yourdata.csv"

_c01 = load("01_cohort")
df_raw = _c01.load_raw(DATA_PATH)
df_raw.head(0)


Loaded: 397 rows × 39 columns


,Nr.,Personas kods,Unnamed: 2,Vecums. gadi,"Dzimums. 0 - vīrietis\n1 - sieviete""",Histoloģija. 0 - nav\n1 - ir,WHO pakāpe (2021). 1 / 2 / 3,Progesterons. 0 - negatīvs\n1 - pozitīvs,Ki-67 (%). skaitlis. %,Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir,Nekroze histoloģiski. 0 - nav\n1 - ir,MRI izmeklējuma datums,Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija,Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base,Cik meningiomas?,Max diametrs. skaitlis.cm,Tilpums,Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT,K/v i/v. 0 - nav\n1 - ir,0 - primārs\n1 - recidīvs,Audzēja robeža. 1 = gluda. \n2 = neregulāra,Dural tail sign. 0 - nav\n1 - ir,Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir,Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna,Perifokāla tūska. 0 - nav\n1 - ir,Perifokālas tūskas tilpums. cm3,Masas efekts. 0 - nav\n1 - ir,Audzēja kalcifikācija. 0 - nav\n1 - ir,Cistiskas komponentes. 0 - nav\n1 - ir,Audzēja nekroze. 0 - nav\n1 - ir,Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams,Kaule hiperostoze. 0 - nav\n1 - ir,Kaula invāzija (cortical destruction). 0 - nav\n1 - ir,Tumor Hyperintensity on DWI. 0 - nav\n1 - ir,Tumor Hyperintensity on T2. 0 - nav\n1 - ir,Tumor Hypointensity on T1. 0 - nav\n1 - ir,Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug,Cauraug falx cerebri 0 - nav. 1 - ir,ADC map value


## 02. Column rename

1. Run **see raw columns** → copy the printed skeleton  
2. Paste into `COLUMN_RENAME_MAP` below and fill in snake_case names  
3. Run **apply rename**


In [3]:
#🟧🟧🟧 Step 1 — see raw columns (run once per new dataset)

load("02_column_rename_map").list_cols(df_raw)


COLUMN_RENAME_MAP = {
    "Nr.": "",
    "Personas kods": "",
    "Unnamed: 2": "",
    "Vecums. gadi": "",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "",
    "Histoloģija. 0 - nav\n1 - ir": "",
    "WHO pakāpe (2021). 1 / 2 / 3": "",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "",
    "Ki-67 (%). skaitlis. %": "",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "",
    "Nekroze histoloģiski. 0 - nav\n1 - ir": "",
    "MRI izmeklējuma datums": "",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "",
    "Cik meningiomas?": "",
    "Max diametrs. skaitlis.cm": "",
    "Tilpums": "",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "",
    "K/v i/v. 0 - nav\n1 - ir": "",
    "0 - primārs\n1 - recidīvs": "",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "",
    "Dural tail sign. 0 - nav\n1 - ir": "",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n

In [4]:
#🟧🟧🟧 Step 2 — paste skeleton here and fill in the right-hand names

COLUMN_RENAME_MAP = {
    "Nr.": "id",
    "Personas kods": "patient_code",
    "Unnamed: 2": "entry_year",
    "Vecums. gadi": "age",
    "Dzimums. 0 - vīrietis\n1 - sieviete\"": "sex",
    "Histoloģija. 0 - nav\n1 - ir": "histology_available",
    "WHO pakāpe (2021). 1 / 2 / 3": "who_grade",
    "Progesterons. 0 - negatīvs\n1 - pozitīvs": "progesterone_pos",
    "Ki-67 (%). skaitlis. %": "ki67_pct",
    "Smadzeņu parenhīmas invāzija. 0 - nav\n1 - ir": "brain_invasion",

    "Nekroze histoloģiski. 0 - nav\n1 - ir": "hist_necrosis",
    "MRI izmeklējuma datums": "mri_date",
    "Puse. 1 - labā\n2 - kreisā\n3 - viduslīnija": "side",
    "Lokalizācija: skull base / non–skull base. 0 - non-skull base\n1 - skull base": "tumor_location",
    "Cik meningiomas?": "meningioma_count",

    "Max diametrs. skaitlis.cm": "max_diameter_cm",
    "Tilpums": "tumor_volume",
    "Pamatmodalitāte analīzei. 0 - MRI\n1 - CT\n3 - MRI+CT": "base_modality",
    "K/v i/v. 0 - nav\n1 - ir": "iv_contrast",
    "0 - primārs\n1 - recidīvs": "tumor_episode",
    "Audzēja robeža. 1 = gluda. \n2 = neregulāra": "tumor_margin",
    "Dural tail sign. 0 - nav\n1 - ir": "dural_tail",
    "Gredzenveida kontrastēšanās (tumor capsular enhancement). 0 - nav\n1 - ir": "capsular_enhancement",
    "Kontrastēšanās veids. 0 - homogēna\n1 - heterogēna": "heterogeneous_enhancement",
    "Perifokāla tūska. 0 - nav\n1 - ir": "perifocal_edema",

    "Perifokālas tūskas tilpums. cm3": "edema_volume_cm3",
    "Masas efekts. 0 - nav\n1 - ir": "mass_effect",
    "Audzēja kalcifikācija. 0 - nav\n1 - ir": "calcification",
    "Cistiskas komponentes. 0 - nav\n1 - ir": "cystic_component",
    "Audzēja nekroze. 0 - nav\n1 - ir": "necrosis",
    "Hemorāģiskas sastāvdaļas. 0 - nav\n1 - ir\n2 - nav skaidri izvērtējams ": "hemorrhage",
    "Kaule hiperostoze. 0 - nav\n1 - ir": "hyperostosis",
    "Kaula invāzija (cortical destruction). 0 - nav\n1 - ir": "cortical_destruction",
    "Tumor Hyperintensity on DWI. 0 - nav\n1 - ir": "dwi_hyperintensity",
    "Tumor Hyperintensity on T2. 0 - nav\n1 - ir": "t2_hyperintensity",
    "Tumor Hypointensity on T1. 0 - nav\n1 - ir": "t1_hypointensity",

    "Sīnuss. 0 - neieaug\n1 - ieaug\n2 - ieaug un cauraug": "sinus_invasion",
    "Cauraug falx cerebri 0 - nav. 1 - ir": "transfalcine_extension",
    "ADC map value": "adc_value",
    }

In [5]:
#🟧🟧🟧 Step 3 — apply rename

df_raw = load("02_column_rename_map").apply_rename(df_raw, COLUMN_RENAME_MAP)
df = df_raw
df.head(0)

,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,base_modality,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value


## 02b. Key columns & cohort filter

Use **renamed** names from §02. Edit, then run.


In [6]:
YEAR_COLUMN = "entry_year"
ID_COLS = ["id", "patient_code", "entry_year"]

ANALYSIS_YEARS: list[int] | None = None   # e.g. [2025]; None = all years
# ANALYSIS_YEARS = 

In [7]:
df_raw = _c01.filter_cohort(df_raw, YEAR_COLUMN, ANALYSIS_YEARS)
df = df_raw

📅 Cohort · all years in entry_year
[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
397 rows


## 03. Schema

1. Run **infer**  
2. Run **print template**  
3. Run **column uniques** (nulls / replace hints)  
4. Edit **schema_overrides**  
5. Run **apply overrides**


In [8]:
schema = infer_schema(df_raw)
schema_summary(schema)


,column,kind,keep,datetime_bin,levels,nulls,note
0,id,id,True,None,None,None,
1,patient_code,id,True,None,None,None,
2,entry_year,nominal,True,None,None,None,
3,age,continuous,True,None,None,None,
4,sex,binary,True,None,None,None,
5,histology_available,binary,True,None,None,None,
6,who_grade,nominal,True,None,None,None,
7,progesterone_pos,binary,True,None,None,None,
8,ki67_pct,text,True,None,None,None,
9,brain_invasion,binary,True,None,None,None,


In [9]:
print_schema_template(schema)


schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind='id'),
    'entry_year': ColSpec(name='entry_year', kind='nominal'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='binary'),
    'histology_available': ColSpec(name='histology_available', kind='binary'),
    'who_grade': ColSpec(name='who_grade', kind='nominal'),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary'),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    'mri_date': ColSpec(name='mri_date', kind='text'),
    'side': ColSpec(name='side', kind='nominal'),
    'tumor_location': ColSpec(name='tumor_location', kind='binary'),
    'meningioma_count': ColSpec(name='meningioma_count', kind='ordinal', ordered_levels=[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]),
    'max_diameter_c

In [10]:
#🟧🟧🟧 inspect raw values — use for nulls=() and replace={} below
print_column_uniques(df_raw, schema)


📋 Column uniques — for nulls=() and replace={} in schema_overrides below

▸ id · id
  · 397 unique
  · 1 → 1
  · 2 → 1
  · 3 → 1
  · 4 → 1
  · 5 → 1
  · 6 → 1
  · 7 → 1
  · 8 → 1
  · … 389 more values

▸ patient_code · id
  · 397 unique
  · '290357-12753' → 1
  · '070458-11352' → 1
  · '230949-11093' → 1
  · '140352-11498' → 1
  · '151269-12200' → 1
  · '270866-10213' → 1
  · '180140-18005' → 1
  · '260962-11211' → 1
  · … 389 more values

▸ entry_year · nominal
  · '2018' → 63
  · '2019' → 57
  · '2020' → 29
  · '2021' → 25
  · '2022' → 22
  · '2023' → 40
  · '2024' → 72
  · '2025' → 83
  · '2026' → 1
  · 'jāpaskatās vēlāk' → 1
  · ∅ → 4

▸ age · continuous
  · 63 unique · 20.0 … 95.0
  · ∅ → 1

▸ sex · binary
  · np.float64(0.0) → 120
  · np.float64(1.0) → 276
  · ∅ → 1

▸ histology_available · binary
  · np.float64(0.0) → 32
  · np.float64(1.0) → 364
  · ∅ → 1

▸ who_grade · nominal
  · '1' → 255
  · '1x operēta Vācijā. tgd inoperabls' → 1
  · '2' → 99
  · '3' → 11
  · 'atteicās no 

In [11]:
#🟧🟧🟧 Edit overrides, then run

schema_overrides = {
    'id': ColSpec(name='id', kind='id'),
    'patient_code': ColSpec(name='patient_code', kind="id", keep=False),
    
    'entry_year': ColSpec(name='entry_year', kind='datetime', keep=False, datetime_bin='year'),
    'age': ColSpec(name='age', kind='continuous'),
    'sex': ColSpec(name='sex', kind='nominal', replace={0:"male", 1:"female",}),
    
    'who_grade': ColSpec(name='who_grade', kind='ordinal', ordered_levels=["1","2","3"]),
    
    'histology_available': ColSpec(name='histology_available', kind='binary', nulls=(2,), keep=False),
    'progesterone_pos': ColSpec(name='progesterone_pos', kind='binary', nulls=(2,)),
    'ki67_pct': ColSpec(name='ki67_pct', kind='text'),
    'brain_invasion': ColSpec(name='brain_invasion', kind='binary'),
    'hist_necrosis': ColSpec(name='hist_necrosis', kind='binary'),
    
    'mri_date': ColSpec(name='mri_date', kind='datetime', keep=False, datetime_bin='full'),
    
    'side': ColSpec(name='side', kind='nominal', replace={'1': "right", '2': "left", '3': "midline"}),
    'tumor_location': ColSpec(name='tumor_location', kind='nominal', replace={0: "non_skull_base", 1: "skull_base"}, nulls=(2,)),
    'meningioma_count': ColSpec(name='meningioma_count', kind='count'),
    'max_diameter_cm': ColSpec(name='max_diameter_cm', kind='continuous'),
    'tumor_volume': ColSpec(name='tumor_volume', kind='continuous'),
    
    'base_modality': ColSpec(name='base_modality', kind='nominal', replace={0: "mri", 1: "ct", 3: "mri_ct"}, keep=False),
    'iv_contrast': ColSpec(name='iv_contrast', kind='binary', keep=False),
    
    'tumor_episode': ColSpec(name='tumor_episode', kind='ordinal', replace={'0': "primary", '1': "recurrent"}, ordered_levels=["primary", "recurrent"], nulls=("multiplas",)),
    
    'tumor_margin': ColSpec(name='tumor_margin', kind='nominal', replace={1: "regular", 2: "irregular"}, nulls=(0,)),
    'dural_tail': ColSpec(name='dural_tail', kind='binary'),
    
    'capsular_enhancement': ColSpec(name='capsular_enhancement', kind='binary'),
    'heterogeneous_enhancement': ColSpec(name='heterogeneous_enhancement', kind='binary'),
    'dwi_hyperintensity': ColSpec(name='dwi_hyperintensity', kind='binary', nulls=('-',)),
    't2_hyperintensity': ColSpec(name='t2_hyperintensity', kind='binary', nulls=('-',)),
    't1_hypointensity': ColSpec(name='t1_hypointensity', kind='binary', nulls=('-',)),
    
    'perifocal_edema': ColSpec(name='perifocal_edema', kind='binary'),
    'edema_volume_cm3': ColSpec(name='edema_volume_cm3', kind='continuous'),
    
    'mass_effect': ColSpec(name='mass_effect', kind='binary'),
    'calcification': ColSpec(name='calcification', kind='binary'),
    'cystic_component': ColSpec(name='cystic_component', kind='binary'),
    'mri_necrosis': ColSpec(name='mri_necrosis', kind='binary'),
    'hemorrhage': ColSpec(name='hemorrhage', kind='binary', nulls=(2.0,)),
    'hyperostosis': ColSpec(name='hyperostosis', kind='binary'),
    'sinus_invasion': ColSpec(name='sinus_invasion', kind='ordinal', replace={0: "no_invasion", 1: "sinus_invasion", 2: "transsinus_extension"}, ordered_levels=["no_invasion", "sinus_invasion", "transsinus_extension"]),
    'cortical_destruction': ColSpec(name='cortical_destruction', kind='binary'),
    'transfalcine_extension': ColSpec(name='transfalcine_extension', kind='binary'),
    
    'adc_value': ColSpec(name='adc_value', kind='continuous'),
    }


In [12]:
load("03_schema_overrides").apply_schema_overrides(schema, schema_overrides, OUTPUT_ROOT)

## 03b. Pre-schema inclusion filters

Apply on **raw strings** before `apply_schema` so excluded rows never become categorical levels (e.g. spinal meningioma in `side` / `mri_date`).

In [13]:
_c04 = load("04_row_filters")

pre_schema_row_filters = [
    _c04.brain_meningioma_row_filter(),
]

df, pre_schema_row_filter_log = _c04.apply_row_filters(
    df, pre_schema_row_filters,
)
n_rows_pre_schema = len(df)
pre_schema_row_filter_log

,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,397,393,4,inclusion criteria - brain meningioma


## 04. Apply schema


In [14]:
schema_log = []
df = apply_schema(df, schema, log=schema_log)
n_rows_after_schema = len(df)
df.head()

,id,patient_code,entry_year,age,sex,histology_available,who_grade,progesterone_pos,ki67_pct,brain_invasion,hist_necrosis,mri_date,side,tumor_location,meningioma_count,max_diameter_cm,tumor_volume,base_modality,iv_contrast,tumor_episode,tumor_margin,dural_tail,capsular_enhancement,heterogeneous_enhancement,perifocal_edema,edema_volume_cm3,mass_effect,calcification,cystic_component,necrosis,hemorrhage,hyperostosis,cortical_destruction,dwi_hyperintensity,t2_hyperintensity,t1_hypointensity,sinus_invasion,transfalcine_extension,adc_value
0,1,290357-12753,2025.0,67.0,female,False,NaN,<NA>,<NA>,<NA>,<NA>,NaT,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,<NA>,NaN
1,2,070458-11352,2025.0,67.0,female,True,1,True,1-3,False,False,2025-06-27,right,skull_base,2.0,4.9,36.50,mri,True,primary,regular,False,True,False,True,5.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,0.88
2,3,230949-11093,2025.0,76.0,female,True,1,True,1-5,False,False,2025-09-05,midline,skull_base,1.0,2.8,6.86,mri_ct,True,primary,irregular,False,True,False,True,26.0,True,False,False,False,False,False,False,True,True,True,no_invasion,True,0.94
3,4,140352-11498,2025.0,73.0,female,True,2,True,25-30,True,True,2025-07-23,right,non_skull_base,1.0,4.7,40.90,mri_ct,True,recurrent,irregular,False,True,True,True,135.0,True,True,True,False,True,False,False,True,True,True,no_invasion,False,0.60
4,5,151269-12200,2025.0,55.0,male,True,1,True,1-2,False,False,2025-08-04,right,non_skull_base,1.0,3.7,8.30,mri_ct,True,primary,irregular,True,True,False,True,24.0,True,True,False,False,False,False,False,True,True,True,no_invasion,False,1.20


## 05. Duplicate audit


In [15]:
dupes, df = audit_duplicates(df, id_cols=ID_COLS, include_first=True, drop=False)
dupes.head() if len(dupes) else print('No duplicate groups found.')

No duplicate groups found.


## 06. Row filters (post-schema)

Edit `post_schema_row_filters` (toggle `active=True/False`), run the cell, then run finalize below.

Pre-schema inclusion filters (§03b) run on raw strings; post-schema filters run after datetime coercion etc.


In [16]:
post_schema_row_filters = [
    _c04.RowFilter(
        name="who_grade exists",
        keep=lambda d: d["who_grade"].notna(),
        note="inclusion criteria - histological WHO grade",
        active=True,
    ),
    _c04.RowFilter(
        name="MRI exists",
        keep=lambda d: d["mri_date"].notna(),
        note="inclusion criteria - MRI",
        active=True,
    ),
    # _c04.RowFilter(
    #     name="sex known",
    #     keep=lambda d: d["sex"] != "unknown",
    #     note="Keep rows where sex is known (not 'unknown')",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="adult patients only",
    #     keep=lambda d: d["age"] >= 18,
    #     note="Keep rows where age is 18 or older",
    #     active=False,
    # ),
    # _c04.RowFilter(
    #     name="exclude WHO grade 2 or 3",
    #     keep=lambda d: ~d["who_grade"].isin(["2", "3"]),
    #     note="Keep rows where WHO grade is 1 (exclude grades 2 and 3)",
    #     active=False,
    # ),
]

df, post_schema_row_filter_log = _c04.apply_row_filters(df, post_schema_row_filters)
row_filter_log = _c04.combine_row_filter_logs(
    pre_schema_row_filter_log,
    post_schema_row_filter_log,
)
row_filter_log


,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,397,393,4,inclusion criteria - brain meningioma
1,who_grade exists,True,393,361,32,inclusion criteria - histological WHO grade
2,MRI exists,True,361,352,9,inclusion criteria - MRI


In [17]:
df = _c04.finalize_row_drops(
    df, row_filter_log,
    output_root=OUTPUT_ROOT,
    df_raw=df_raw,
    n_rows_pre_schema=n_rows_pre_schema,
    n_rows_after_schema=n_rows_after_schema,
    schema=schema,
    dupes=dupes,
    schema_log=schema_log,
)


,name,active,rows_before,rows_after,rows_removed,note
0,Meningioma location - brain,True,397,393,4,inclusion criteria - brain meningioma
1,who_grade exists,True,393,361,32,inclusion criteria - histological WHO grade
2,MRI exists,True,361,352,9,inclusion criteria - MRI


## 07. DDA — first pass


In [18]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))



--- overall ---


,n_rows,n_cols,n_cols_analysed,missing_cells_pct
0,352,39,33,0.7



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,352,60,0.0,20.00,40.00,65.00,63.11,63.70,81.00,92.0,71.00,12.68,0.20,17.25,-0.43,-0.14
1,meningioma_count,count,352,6,0.0,1.00,1.00,1.00,1.17,1.02,2.00,6.0,1.00,0.58,0.50,0.00,4.51,24.42
2,max_diameter_cm,continuous,352,88,0.0,0.20,1.79,3.80,4.06,3.95,7.30,9.2,1.80,1.73,0.43,2.50,0.51,-0.42
3,tumor_volume,continuous,329,282,6.5,0.30,1.60,14.70,27.62,21.57,98.76,168.0,2.00,31.80,1.15,31.42,1.66,2.31
4,edema_volume_cm3,continuous,333,182,5.4,0.00,0.00,4.48,20.85,13.43,93.60,197.0,0.00,32.74,1.57,29.20,2.16,5.06
5,adc_value,continuous,309,77,12.2,0.41,0.64,0.82,0.85,0.83,1.19,1.7,0.79,0.17,0.20,0.17,1.19,2.99



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,rarest_pct,max_class_imbalance,median_category,balance,entropy_bin
0,sex,nominal,False,352,2,0,female,69.0,,,male,31.0,2.23,,0.89,0.89
1,who_grade,ordinal,True,352,3,0,1,70.2,2,26.7,3,3.1,22.45,1,0.65,1.02
2,side,nominal,False,352,3,0,right,45.2,left,44.9,midline,9.9,4.54,,0.86,1.37
3,tumor_location,nominal,False,352,2,0,non_skull_base,54.5,,,skull_base,45.5,1.20,,0.99,0.99
4,tumor_episode,ordinal,True,352,2,0,primary,88.1,,,recurrent,11.9,7.38,primary,0.53,0.53
5,tumor_margin,nominal,False,352,2,0,regular,55.4,,,irregular,44.6,1.24,,0.99,0.99
6,sinus_invasion,ordinal,True,352,3,0,no_invasion,74.4,sinus_invasion,18.2,transsinus_extension,7.4,10.08,no_invasion,0.66,1.04



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,mode,mode_pct,rarest,rarest_pct,max_class_imbalance,balance,entropy_bin
0,progesterone_pos,binary,False,351,2,0.3,True,97.4,False,2.6,38.00,0.17,0.17
1,brain_invasion,binary,False,352,2,0.0,False,98.3,True,1.7,57.67,0.12,0.12
2,hist_necrosis,binary,False,352,2,0.0,False,90.1,True,9.9,9.06,0.47,0.47
3,dural_tail,binary,False,352,2,0.0,True,81.2,False,18.8,4.33,0.70,0.70
4,capsular_enhancement,binary,False,352,2,0.0,True,88.1,False,11.9,7.38,0.53,0.53
5,heterogeneous_enhancement,binary,False,352,2,0.0,True,58.8,False,41.2,1.43,0.98,0.98
6,perifocal_edema,binary,False,352,2,0.0,True,65.3,False,34.7,1.89,0.93,0.93
7,mass_effect,binary,False,352,2,0.0,True,86.1,False,13.9,6.18,0.58,0.58
8,calcification,binary,False,352,2,0.0,False,53.1,True,46.9,1.13,1.00,1.00
9,cystic_component,binary,False,352,2,0.0,False,78.7,True,21.3,3.69,0.75,0.75



--- datetime ---
(none)

--- id_text ---


,column,kind,n,missing_pct,n_unique
0,id,id,352,0,352
1,ki67_pct,text,352,0,42


## 08. Missingness analysis


In [19]:
missing_summary = analyze_missingness(df, output_root=OUTPUT_ROOT)
missing_summary[missing_summary.n_missing > 0]

,column,n_missing,pct_missing
0,adc_value,43,12.22
1,tumor_volume,23,6.53
2,edema_volume_cm3,19,5.40
3,dwi_hyperintensity,4,1.14
4,hemorrhage,3,0.85
5,t2_hyperintensity,2,0.57
6,t1_hypointensity,2,0.57
7,hyperostosis,1,0.28
8,cortical_destruction,1,0.28
9,progesterone_pos,1,0.28


### 08a. Missingness policy

Declare structural and MNAR decisions in lists below (same pattern as row filters).
Run the next cell to apply them once and get an audit log.

- **Structural** — NaN means the slot does not exist (do not impute; derive count/max instead).
- **MNAR** — missingness itself may be informative (adds `<col>_missing` flag).


In [20]:
_c05 = load("05_missingness")

STRUCTURAL_GROUPS = [
    # Example only. Keep empty if we do not currently have slot-style columns.
    # _c05.StructuralGroup(
    #     name="lesion_mri_pirads",
    #     cols=["lesion_1_MRI_PIRADS", "lesion_2_MRI_PIRADS", "lesion_3_MRI_PIRADS"],
    #     derive_count_col="n_mri_pirads_lesions",
    #     derive_max_col="max_mri_pirads",
    #     skip_raw=True,
    #     reason="Blank lesion slots mean lesion does not exist, not unknown.",
    # ),
]

MNAR_COLUMNS = [
    # Add only when missingness itself may be informative.
    # _c05.MnarColumn(
    #     col="ki67_pct",
    #     flag_col="ki67_pct_missing",
    #     reason="Ki-67 may be absent because it was not measured/reported in selected cases.",
    # ),
    # _c05.MnarColumn(
    #     col="adc_value",
    #     flag_col="adc_value_missing",
    #     reason="ADC may be absent when DWI/ADC was unavailable or non-diagnostic.",
    # ),
]


In [21]:
df, schema, missingness_log = _c05.apply_missingness_policy(
    df=df,
    schema=schema,
    structural_groups=STRUCTURAL_GROUPS,
    mnar_columns=MNAR_COLUMNS,
)

missingness_log

,policy,name,requested_cols,available_cols,missing_cols,created_cols,schema_action,reason,status


## 09. Derivations

Declare derived columns in a list below (same pattern as row filters / missingness).
All study-specific logic lives in the notebook; `06_derivations.py` is just the engine.

**Building blocks:**

- **`BinNumeric`** — cut a numeric column into ordered bins (`age` → `age_bins`).
  Bins = edge values; labels = one per gap. `len(bins) - 1 == len(labels)`.
  Default `right=False`: left-closed intervals (`[50, 60)` → `"50-59"`).
- **`Apply`** — custom logic via helper + `fn=lambda s: ...` (Ki-67 midpoint, grouped labels, boolean flags, etc.).

Each entry supports `active=False` (skip) and `overwrite=True` (replace existing column).
Append to `DERIVATIONS` to add columns — no `.py` edits needed.


In [22]:
_c06 = load("06_derivations")


def _ki67_midpoint(x):
    if pd.isna(x):
        return pd.NA
    parts = str(x).replace(",", ".").split("-")
    nums = [float(p) for p in parts]
    return sum(nums) / len(nums)
def _ki67_group(x):
    if pd.isna(x):
        return pd.NA
    if x <= 4:
        return "low_le_4"
    if x < 10:
        return "intermediate_5_9"
    return "high_ge_10"

DERIVATIONS = [
    _c06.BinNumeric(
        name="age_bins",
        source="age",
        bins=[-np.inf, 50, 60, 70, 80, np.inf],
        labels=["<50", "50-59", "60-69", "70-79", "80+"],
        kind="ordinal",
        active=True,
        overwrite=False,
        reason="Age groups for descriptive tables.",
    ),
    _c06.Apply(
        name="high_grade",
        source="who_grade",
        fn=lambda s: s.astype("Float64").pipe(lambda sf: (sf == 2) | (sf == 3)),
        kind="binary",
        active=True,
        overwrite=False,
        reason="WHO grade 2/3 = high-grade meningioma.",
    ),
    _c06.Apply(
        name="multiple_meningiomas",
        source="meningioma_count",
        fn=lambda s: s.astype("Float64") > 1,
        kind="binary",
        active=True,
        overwrite=False,
        reason=">1 meningioma = multiple",
    ),
    _c06.Apply(
        name="ki67_mid",
        source="ki67_pct",
        fn=lambda s: s.map(_ki67_midpoint).astype("Float64"),
        kind="continuous",
        active=True,
        overwrite=False,
        reason="Midpoint of Ki-67 range strings.",
    ),
    _c06.Apply(
        name="ki67_group",
        source="ki67_mid",
        fn=lambda s: s.map(_ki67_group),
        kind="ordinal",
        ordered_levels=["low_le_4", "intermediate_5_9", "high_ge_10"],
        active=True,
        overwrite=False,
        reason="Ki-67 clinical groups: ≤4 / 5-9 / ≥10.",
    ),
    _c06.Compute(
        name="edema_volume_cm3",
        sources=["perifocal_edema", "edema_volume_cm3"],
        fn=lambda d: d["edema_volume_cm3"].mask(
            d["perifocal_edema"].fillna(True).astype(float) == 0, 0
        ),
        kind="continuous",
        active=True,
        overwrite=True,
        reason="No perifocal edema => edema volume is structurally 0, not missing.",
    ),
    ]


In [23]:
df, schema, derivation_log = _c06.apply_derivations(
    df=df,
    schema=schema,
    derivations=DERIVATIONS,
    output_root=OUTPUT_ROOT,
    write_csv=True,
)

derivation_log

,age_bins,high_grade,multiple_meningiomas,ki67_mid,ki67_group
1,60-69,False,True,2.0,low_le_4
2,70-79,False,False,3.0,low_le_4
3,70-79,True,False,27.5,high_ge_10
4,50-59,False,False,1.5,low_le_4
5,50-59,False,False,1.5,low_le_4


,derivation,type,active,source,kind,rows_nonmissing,rows_missing,schema_action,warning,reason
0,age_bins,BinNumeric,True,age,ordinal,352,0,added ColSpec (ordinal) for age_bins,,Age groups for descriptive tables.
1,high_grade,Apply,True,who_grade,binary,352,0,added ColSpec (binary) for high_grade,,WHO grade 2/3 = high-grade meningioma.
2,multiple_meningiomas,Apply,True,meningioma_count,binary,352,0,added ColSpec (binary) for multiple_meningiomas,,>1 meningioma = multiple
3,ki67_mid,Apply,True,ki67_pct,continuous,352,0,added ColSpec (continuous) for ki67_mid,,Midpoint of Ki-67 range strings.
4,ki67_group,Apply,True,ki67_mid,ordinal,352,0,added ColSpec (ordinal) for ki67_group,,Ki-67 clinical groups: ≤4 / 5-9 / ≥10.
5,edema_volume_cm3,Compute,True,"perifocal_edema, edema_volume_cm3",continuous,333,19,updated ColSpec (continuous) for edema_volume_cm3,,No perifocal edema => edema volume is structur...


## 10. DDA — second pass


In [24]:
from cleaning import format_table_for_display
from IPython.display import display

dda_tables = run_dda(df, schema, output_root=OUTPUT_ROOT)

for name in ("overall", "continuous", "categorical", "binary", "datetime", "id_text"):
    print(f"\n--- {name} ---")
    tbl = dda_tables.get(name)
    if tbl is None or tbl.empty:
        print("(none)")
    else:
        display(format_table_for_display(tbl))



--- overall ---


,n_rows,n_cols,n_cols_analysed,missing_cells_pct
0,352,44,38,0.6



--- continuous ---


,column,kind,n,n_unique,missing_pct,min,p_5th,median,mean,trimmed_mean,p_95th,max,mode,std,cv,iqr,skewness,kurtosis
0,age,continuous,352,60,0.0,20.00,40.00,65.00,63.11,63.70,81.00,92.0,71.00,12.68,0.20,17.25,-0.43,-0.14
1,meningioma_count,count,352,6,0.0,1.00,1.00,1.00,1.17,1.02,2.00,6.0,1.00,0.58,0.50,0.00,4.51,24.42
2,max_diameter_cm,continuous,352,88,0.0,0.20,1.79,3.80,4.06,3.95,7.30,9.2,1.80,1.73,0.43,2.50,0.51,-0.42
3,tumor_volume,continuous,329,282,6.5,0.30,1.60,14.70,27.62,21.57,98.76,168.0,2.00,31.80,1.15,31.42,1.66,2.31
4,edema_volume_cm3,continuous,333,182,5.4,0.00,0.00,4.48,20.85,13.43,93.60,197.0,0.00,32.74,1.57,29.20,2.16,5.06
5,adc_value,continuous,309,77,12.2,0.41,0.64,0.82,0.85,0.83,1.19,1.7,0.79,0.17,0.20,0.17,1.19,2.99
6,ki67_mid,continuous,352,29,0.0,1.00,1.00,2.50,4.34,2.97,17.50,55.0,1.00,5.88,1.35,3.00,3.92,21.29



--- categorical ---


,column,kind,ordered,n,n_unique,missing_pct,first_mode,first_mode_pct,second_mode,second_mode_pct,rarest,rarest_pct,max_class_imbalance,median_category,balance,entropy_bin
0,sex,nominal,False,352,2,0,female,69.0,,,male,31.0,2.23,,0.89,0.89
1,who_grade,ordinal,True,352,3,0,1,70.2,2,26.7,3,3.1,22.45,1,0.65,1.02
2,side,nominal,False,352,3,0,right,45.2,left,44.9,midline,9.9,4.54,,0.86,1.37
3,tumor_location,nominal,False,352,2,0,non_skull_base,54.5,,,skull_base,45.5,1.20,,0.99,0.99
4,tumor_episode,ordinal,True,352,2,0,primary,88.1,,,recurrent,11.9,7.38,primary,0.53,0.53
5,tumor_margin,nominal,False,352,2,0,regular,55.4,,,irregular,44.6,1.24,,0.99,0.99
6,sinus_invasion,ordinal,True,352,3,0,no_invasion,74.4,sinus_invasion,18.2,transsinus_extension,7.4,10.08,no_invasion,0.66,1.04
7,age_bins,ordinal,True,352,5,0,60-69,28.7,70-79,26.7,80+,8.2,3.48,60-69,0.95,2.21
8,ki67_group,ordinal,True,352,3,0,low_le_4,74.2,intermediate_5_9,15.9,high_ge_10,9.9,7.46,low_le_4,0.68,1.07



--- binary ---


,column,kind,ordered,n,n_unique,missing_pct,mode,mode_pct,rarest,rarest_pct,max_class_imbalance,balance,entropy_bin
0,progesterone_pos,binary,False,351,2,0.3,True,97.4,False,2.6,38.00,0.17,0.17
1,brain_invasion,binary,False,352,2,0.0,False,98.3,True,1.7,57.67,0.12,0.12
2,hist_necrosis,binary,False,352,2,0.0,False,90.1,True,9.9,9.06,0.47,0.47
3,dural_tail,binary,False,352,2,0.0,True,81.2,False,18.8,4.33,0.70,0.70
4,capsular_enhancement,binary,False,352,2,0.0,True,88.1,False,11.9,7.38,0.53,0.53
5,heterogeneous_enhancement,binary,False,352,2,0.0,True,58.8,False,41.2,1.43,0.98,0.98
6,perifocal_edema,binary,False,352,2,0.0,True,65.3,False,34.7,1.89,0.93,0.93
7,mass_effect,binary,False,352,2,0.0,True,86.1,False,13.9,6.18,0.58,0.58
8,calcification,binary,False,352,2,0.0,False,53.1,True,46.9,1.13,1.00,1.00
9,cystic_component,binary,False,352,2,0.0,False,78.7,True,21.3,3.69,0.75,0.75



--- datetime ---
(none)

--- id_text ---


,column,kind,n,missing_pct,n_unique
0,id,id,352,0,352
1,ki67_pct,text,352,0,42


## 11. Analysis targets & predictors

Edit lists, then run.

- **`INFERENTIAL_MODEL_VARIANTS`** — named multivariable calculator specs: `(id, title, link, target, [predictors])`. `link` is shown as a clickable **source** in the HTML report (use `""` if none). Each variant gets its own outcome, model, EPV bar, forest plot, VIF, and report block.


In [25]:
#df.columns.to_list()

In [26]:
EDA_TARGETS = ['high_grade', 'progesterone_pos', 'brain_invasion', 'ki67_group', 'hist_necrosis']
EDA_PREDICTORS = [
    'age',
    'age_bins',
    'sex',
    
    #'who_grade', ==> TARGET
    #'high_grade', ==> TARGET
    
    #'progesterone_pos',
    #'ki67_pct',
    #'ki67_mid',
    #'ki67_group'
    #'brain_invasion',
    #'hist_necrosis',
    
    'side',
    'tumor_location',
    'meningioma_count',
    'multiple_meningiomas',
    'max_diameter_cm',
    'tumor_volume',
    
    'tumor_episode',
    'tumor_margin',
    'dural_tail',
    
    'perifocal_edema',
    'edema_volume_cm3',
    
    'mass_effect',
    'calcification',
    'cystic_component',
    'necrosis',
    'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    'sinus_invasion',
    'transfalcine_extension',
    
    'capsular_enhancement',
    'heterogeneous_enhancement',
    'dwi_hyperintensity',
    't2_hyperintensity',
    't1_hypointensity',
    
    'adc_value',
    ]

INFERENTIAL_TARGETS = ['high_grade']
INFERENTIAL_PREDICTORS = [
    #'age',
    #'age_bins',

    #'sex',
    #'side',
    #'tumor_location',

    #'meningioma_count',
    #'multiple_meningiomas',

    #'max_diameter_cm',
    'tumor_volume',
    #'tumor_episode',
    'tumor_margin',
    'dural_tail',
    #'capsular_enhancement',
    #'heterogeneous_enhancement',

    #'perifocal_edema',
    'edema_volume_cm3',

    'mass_effect',
    #'calcification',
    'cystic_component',
    #'mri_necrosis',
    #'hemorrhage',
    'hyperostosis',
    'cortical_destruction',
    #'dwi_hyperintensity',
    #'t2_hyperintensity',
    #'t1_hypointensity',
    #'sinus_invasion',
    #'transfalcine_extension',
    'adc_value',
    ]

#🟧🟧🟧 Multivariable model variants — each gets its own EPV bar, forest plot, VIF, table, interpretation.
# Use (id, title, link, target, [predictors]) or {"id": ..., "title": ..., "link": ..., "target": ..., "predictors": [...]}.
INFERENTIAL_MODEL_VARIANTS = [
    ("experimental", "meningioma_atypier experimental", "", "high_grade", INFERENTIAL_PREDICTORS),

    # Research work: Predicting the grade of meningiomas by clinical–radiological features: A comparison of precontrast and postcontrast MRI
    # Authors: Yuan Yao, Yifan Xu, Shihe Liu, Feng Xue, Bao Wang, Shanshan Qin, Xiubin Sun, Jingzhen He
    # Link: https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full
    (
        "yao_et_al_2022",
        "Yao et al. 2022 | precontrast / semantic MRI model",
        "https://www.frontiersin.org/journals/oncology/articles/10.3389/fonc.2022.1053089/full",
        "high_grade",
        [
            "sex",
            "tumor_margin",
            "cystic_component",
            "perifocal_edema",
            "dural_tail",
        ],
    ),

    # Research work: Preoperative Prediction of Intracranial Meningioma Grade Using Conventional CT and MRI
    # Authors: T. Amano et al.
    # Link: https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri
    (
        "amano_et_al_2021_expanded_proxy",
        "Amano et al. 2021 expanded proxy | conventional CT/MRI + tumor burden",
        "https://www.cureus.com/articles/80763-preoperative-prediction-of-intracranial-meningioma-grade-using-conventional-ct-and-mri",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "heterogeneous_enhancement",
            "perifocal_edema",
            "tumor_volume",
            "cortical_destruction",
        ],
    ),

    # Research work: The Role of Pre-Operative MRI for Prediction of High-Grade Intracranial Meningioma: A Retrospective Study
    # Authors: Kan Radeesri, Vitit Lekhavat
    # Link: https://journal.waocp.org/article_90552.html
    (
        "radeesri_lekhavat_2020",
        "Radeesri & Lekhavat 2020 | edema / necrosis MRI model",
        "https://journal.waocp.org/article_90552.html",
        "high_grade",
        [
            "perifocal_edema",
            "edema_volume_cm3",
            "mri_necrosis",
            "hemorrhage",
            "hyperostosis",
            "mass_effect",
        ],
    ),

    # Research work: Role of ADC values and ratios of MRI scan in differentiating typical, atypical and anaplastic meningiomas
    # Authors: M. Azeemuddin et al.
    # Link: https://pubmed.ncbi.nlm.nih.gov/30317276/
    (
        "azeemuddin_et_al_2018",
        "Azeemuddin et al. 2018 | diffusion-augmented MRI model",
        "https://pubmed.ncbi.nlm.nih.gov/30317276/",
        "high_grade",
        [
            "adc_value",
            "dwi_hyperintensity",
            "tumor_location",
            "tumor_margin",
            "perifocal_edema",
            "heterogeneous_enhancement",
            "sex",
        ],
    ),

    # Research work: Diagnostic nomogram model for predicting preoperative pathological grade of meningioma
    # Authors: Shijun Peng, Zhihua Cheng, Zhilin Guo
    # Link: https://tcr.amegroups.org/article/view/55552/html
    (
        "peng_cheng_guo_2021",
        "Peng, Cheng & Guo 2021 | interface / invasion model",
        "https://tcr.amegroups.org/article/view/55552/html",
        "high_grade",
        [
            "tumor_location",
            "tumor_margin",
            "sinus_invasion",
            "cortical_destruction",
            "mass_effect",
            "edema_volume_cm3",
            "hyperostosis",
        ],
    ),
]

In [27]:
_c07 = load("07_analysis")
(
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
    EDA_POSITIVE_CLASS,
    INFERENTIAL_POSITIVE_CLASS,
) = _c07.resolve_analysis(
    df,
    EDA_TARGETS,
    EDA_PREDICTORS,
    INFERENTIAL_TARGETS,
    INFERENTIAL_PREDICTORS,
)
INFERENTIAL_MODEL_VARIANTS = _c07.resolve_inferential_variants(
    df, INFERENTIAL_MODEL_VARIANTS,
)

## 10. EDA — univariate screening

Targets can be **binary**, **continuous**, **ordinal**, or **nominal** (from schema). The test depends on both outcome and predictor types — e.g. ordinal outcome × nominal predictor → χ²; continuous outcome × nominal predictor → Kruskal–Wallis.

| target kind   | continuous / count predictor | ordinal predictor | nominal / binary predictor |
|---------------|------------------------------|-------------------|----------------------------|
| binary        | Mann–Whitney U               | Spearman ρ        | χ² / Fisher                |
| continuous    | Spearman ρ                   | Spearman ρ        | Kruskal–Wallis             |
| ordinal       | Spearman ρ                   | Spearman ρ        | χ²                         |
| nominal       | Kruskal–Wallis               | χ²                | χ²                         |

`POSITIVE_CLASS` applies only to **binary** targets. Multivariable logistic (§11) remains **binary outcomes only**.

Per-target p-values are corrected with **Benjamini–Hochberg FDR**.


In [28]:
assoc = screen_associations(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
    )

diag_acc = screen_diagnostic_accuracy(
    df, schema,
    targets=EDA_TARGETS,
    predictors=EDA_PREDICTORS,
    positive_class=EDA_POSITIVE_CLASS,
    fdr_alpha=0.05,
    output_root=OUTPUT_ROOT,
)

#assoc[assoc['fdr_significant']]

C:\Users\UltimateFamily\AppData\Local\Temp\ipykernel_12912\2131924962.py:10: UserWarning: Diagnostic accuracy skipped target 'ki67_group': requires binary outcome (kind=ordinal).
  diag_acc = screen_diagnostic_accuracy(


In [29]:
#🟧🟧🟧 Full table
#assoc

## 11. Multiple imputation (MICE)

We generate **m=10** imputed datasets via sklearn's IterativeImputer
(RandomForest estimator, separate random seed per imputation).
The pooled inferential stage applies Rubin's rules over these 10 fits.

For a quick screening run set `m=3`. For publication use `m≥10`.


In [30]:
# --- MICE settings (pick one profile) ---
# Fast iteration:     m=3,  max_iter=10, n_estimators=20
# Publication run:    m=10, max_iter=25, n_estimators=50

imputed_frames = mice_impute(
    df,
    schema,
    m=10,                # fast: 3  |  publication: 10
    max_iter=25,         # fast: 10 |  publication: 25
    n_estimators=50,     # fast: 20 |  publication: 50
    random_state=42,
    output_root=OUTPUT_ROOT,
)

🧊 MICE starting — 10 imputations, 99 coded missing cells across 40 columns × 352 rows
⚙️  MICE parallel settings:
   🖥️  OS detected: Windows
   🔋 macOS on battery: False
   🧮 CPU count: 16
   🛡️  available cores (after safety margin): 14
   📦 m imputations: 10
   🔀 n_jobs_imputations: 4
   🌲 n_jobs_rf: 3
   🎰 estimated worker slots: 12
   🚦 max_worker_slots: 12
   🔧 backend: loky
   🆘 emergency_safe_mode: False
🚀 Launching 10 imputations across 4 parallel workers…
🏁 MICE complete — 10 frames in 2m 32s (15.2s avg per draw)
📊 NaN count in first imputed frame: 0


## 12. Multivariable logistic regression (Rubin-pooled)

For each **model variant** (see `INFERENTIAL_MODEL_VARIANTS` in §11 — each row specifies its own target):

1. Build design matrix (continuous z-scored, ordinal kept as codes, nominal one-hot).
2. Iteratively drop predictors with **VIF > 5** to handle collinearity.
3. Fit logistic regression on each of the m imputed frames.
4. Pool coefficients with **Rubin's rules** (Barnard–Rubin df).
5. Report adjusted OR with 95% CI and pooled p-value.
6. Save a forest plot SVG, VIF table, calculator JSON, and report section per variant.


In [31]:
#🟧🟧🟧 Skip MICE — median/mode imputation (binary left NaN by default)

#_df_pre_impute = df.copy()
#imputed_frames = [simple_impute(df, schema, impute_binary=False)]

#display(imputation_audit(
#    _df_pre_impute,
#    imputed_frames[0],
#    schema,
#    INFERENTIAL_PREDICTORS,
#    impute_binary=False,
#))

#print("NaN count (all columns):", imputed_frames[0].isna().sum().sum())

In [32]:
display(summarize_multivariable_cases(
    imputed_frames[0],
    schema,
    targets=INFERENTIAL_TARGETS,
    variants=INFERENTIAL_MODEL_VARIANTS,
    positive_class=INFERENTIAL_POSITIVE_CLASS,
    vif_threshold=5.0,
))

,target,model_id,model_title,model_link,n_rows_total,n_complete_cases,n_rows_dropped,n_outcome_events,n_design_columns,epv
0,high_grade,experimental,meningioma_atypier experimental,,352,352,0,105,9,11.7
1,high_grade,yao_et_al_2022,Yao et al. 2022 | precontrast / semantic MRI m...,https://www.frontiersin.org/journals/oncology/...,352,352,0,105,5,21.0
2,high_grade,amano_et_al_2021_expanded_proxy,Amano et al. 2021 expanded proxy | conventiona...,https://www.cureus.com/articles/80763-preopera...,352,352,0,105,6,17.5
3,high_grade,radeesri_lekhavat_2020,Radeesri & Lekhavat 2020 | edema / necrosis MR...,https://journal.waocp.org/article_90552.html,352,352,0,105,5,21.0
4,high_grade,azeemuddin_et_al_2018,Azeemuddin et al. 2018 | diffusion-augmented M...,https://pubmed.ncbi.nlm.nih.gov/30317276/,352,352,0,105,7,15.0
5,high_grade,peng_cheng_guo_2021,"Peng, Cheng & Guo 2021 | interface / invasion ...",https://tcr.amegroups.org/article/view/55552/html,352,352,0,105,7,15.0


In [33]:
from statsmodels.tools.sm_exceptions import ConvergenceWarning as SMConvergenceWarning

with warnings.catch_warnings():
    warnings.simplefilter("ignore", SMConvergenceWarning)
    inf_results = run_inferential(
        imputed_frames, schema,
        targets=INFERENTIAL_TARGETS,
        variants=INFERENTIAL_MODEL_VARIANTS,
        positive_class=INFERENTIAL_POSITIVE_CLASS,
        vif_threshold=5.0,
        output_root=OUTPUT_ROOT,
    )
#inf_results

## 12. Report (§08)

Builds `report.html` from artifacts already in `output/` (DDA, EDA, inferential).
Edit the settings cell, then run both cells.

- **`REPORT_TITLE` / `REPORT_AUTHOR`** — shown on the cover.
- **`REPORT_PATH`** — where to write the HTML file.
- **`analysis_years`** — optional cohort label suffix on the title (from §02b).


In [34]:
REPORT_TITLE = "REPORT: Non-invasive radiological biomarkers of meningiomas as a prognostic tool for predicting tumor histological grade"
REPORT_AUTHOR = "Andris & the radio team"
REPORT_PATH = OUTPUT_ROOT / "report" / "report.html"

In [35]:
_c08 = load("08_report_settings")
_c08.run_report(
    output_root=OUTPUT_ROOT,
    report_title=REPORT_TITLE,
    report_author=REPORT_AUTHOR,
    report_path=REPORT_PATH,
    analysis_years=ANALYSIS_YEARS,
    eda_targets=EDA_TARGETS,
)

Report written: C:\dev\TheLibraryOfCode\meningioma-atypier\heavy_machinery\output\report\report.html


WindowsPath('output/report/report.html')

## 13. Outputs

Everything is saved under `output/`.


In [36]:
from pathlib import Path
for p in sorted(Path(OUTPUT_ROOT).rglob('*')):
    if p.is_file():
        print(p)

output\cleaning\cleaned.csv
output\cleaning\cleaning_log.csv
output\cleaning\cleaning_summary.csv
output\cleaning\derivation_log.csv
output\dda\figures\adc_value__box.svg
output\dda\figures\adc_value__hist.svg
output\dda\figures\age__box.svg
output\dda\figures\age__hist.svg
output\dda\figures\age_bins__bar.svg
output\dda\figures\brain_invasion__bar.svg
output\dda\figures\calcification__bar.svg
output\dda\figures\capsular_enhancement__bar.svg
output\dda\figures\cortical_destruction__bar.svg
output\dda\figures\cystic_component__bar.svg
output\dda\figures\dural_tail__bar.svg
output\dda\figures\dwi_hyperintensity__bar.svg
output\dda\figures\edema_volume_cm3__box.svg
output\dda\figures\edema_volume_cm3__hist.svg
output\dda\figures\hemorrhage__bar.svg
output\dda\figures\heterogeneous_enhancement__bar.svg
output\dda\figures\high_grade__bar.svg
output\dda\figures\hist_necrosis__bar.svg
output\dda\figures\hyperostosis__bar.svg
output\dda\figures\ki67_group__bar.svg
output\dda\figures\ki67_mid__